# 01 — Reproduction: vanilla TabPFN on NSL-KDD

Walks through loading, preprocessing and a single-context TabPFN run.

The notebook imports the package and calls it; no pipeline logic lives here, so what you see is the same tested code the scripts run.

**Runtime:** about 3 minutes on an M1 with `n_estimators=2` and a 5,000-row test sample.

In [ ]:
import numpy as np
import pandas as pd

from tabpfn_nids import config
from tabpfn_nids.data_pipeline import load_nsl_kdd, load_and_preprocess_nsl_kdd
from tabpfn_nids.models import TabPFNWrapper
from tabpfn_nids.evaluation import compute_metrics, format_metrics, plot_all

config.setup_logging()
config.set_seed(config.SEED)
print('checkpoint:', config.TABPFN_CHECKPOINT)
print('device    :', config.resolve_device())

## 1. Load the raw data

NSL-KDD ships headerless. Column names come from the `@attribute` lines in `KDDTrain+.arff`, which is in the same archive — read from the dataset, not from memory.

In [ ]:
train_df, test_df = load_nsl_kdd()
print(train_df.shape, test_df.shape)
train_df.head()

### The test split is the point of the benchmark

It contains attack types that never appear in training. That is what makes NSL-KDD a test of *detecting novel attacks* rather than of memorisation.

In [ ]:
train_attacks = set(train_df['attack'].unique())
test_attacks = set(test_df['attack'].unique())
unseen = sorted(test_attacks - train_attacks)
print(f'{len(train_attacks)} attack labels in train, {len(test_attacks)} in test')
print(f'{len(unseen)} unseen in test: {unseen}')

## 2. Preprocess

One-hot the three nominal columns, standardise the 38 numeric ones, binarise the label. Every transformer is fitted on **train only**.

`difficulty` is dataset metadata, not a feature — it is dropped. Feeding it to a model would leak.

In [ ]:
X_train_full, y_train_full, X_test, y_test = load_and_preprocess_nsl_kdd()
print(f'X_train {X_train_full.shape}   X_test {X_test.shape}')
print(f'attack rate: train {y_train_full.mean():.2%}, test {y_test.mean():.2%}')

## 3. Subsample to one TabPFN context

TabPFN accepts at most 10,000 in-context samples, so the baseline must discard 92% of the training data. Lifting that is what Enhancement 1 is for.

The subsample is stratified to preserve the 53/47 class balance.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, train_size=10_000, random_state=config.SEED)
idx, _ = next(splitter.split(X_train_full, y_train_full))
X_train, y_train = X_train_full[idx], y_train_full[idx]

# Subsample the test set too, to keep the notebook to a few minutes.
t_idx, _ = next(StratifiedShuffleSplit(n_splits=1, train_size=5_000,
                random_state=config.SEED).split(X_test, y_test))
X_test_s, y_test_s = X_test[t_idx], y_test[t_idx]
print(f'context {X_train.shape}  (attack rate {y_train.mean():.2%})')

## 4. Fit and predict

TabPFN's `fit` only caches the context — it takes under a second. Essentially all the compute is in `predict`.

In [ ]:
model = TabPFNWrapper(random_state=config.SEED, n_estimators=2)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test_s)
y_pred = np.argmax(y_proba, axis=1)
print(f'fit {model.fit_seconds:.2f}s  predict {model.predict_seconds:.1f}s')

## 5. Results

In [ ]:
metrics = compute_metrics(y_test_s, y_pred, y_proba)
print(format_metrics(metrics, title='NSL-KDD baseline'))

### Reading this honestly

Precision is high and recall is low. The model is right when it flags an attack but misses about a third of them — the expected consequence of the 17 unseen attack types.

ROC-AUC near 0.95 against F1 near 0.75 says the *ranking* is good and the 0.5 threshold is simply badly placed. The PR curve below makes that visible.

In [ ]:
paths = plot_all(y_test_s, y_proba, metrics['confusion_matrix'], prefix='nb01_baseline')
for name, path in paths.items():
    print(f'{name}: {path}')

from IPython.display import Image, display
for path in paths.values():
    display(Image(str(path)))